# Atividade 1 — Notebook inicial: tokenizador e dataloaders

Este notebook segue a infraestrutura original da Atividade 1 e a implementação de Sebastian Raschka em *Build a Large Language Model (From Scratch)*. Na Semana 1 da graduação, vamos treinar um BPE próprio, treinar o GPT-2 vanilla e medir loss, perplexidade e BPB em validação.

## 1. Instalação e configuração

Para treinar, selecione uma GPU no Colab, coloque o ZIP da disciplina em **Meu Drive** e clique em **Executar tudo**. As seções definem cada etapa do roteiro. A célula em **10. Conclusão da Semana 1** executa essas etapas na ordem.

In [ ]:
ZIP_NO_DRIVE = "corpus_ufcg_introducoes_alunos.zip"
PASTA_PROJETO_NO_DRIVE = "Atividade1_UFCG"

CONFIG_VANILLA = {
    "vocab_size": 16_000,
    "context_length": 256,
    "emb_dim": 256,
    "n_heads": 4,
    "n_layers": 4,
    "drop_rate": 0.1,
    "qkv_bias": False,
}
SEMENTE = 123
LOTE = 32
INTERVALO_CHECKPOINT = 500
LR_INICIAL, LR_PICO, LR_MINIMO = 3e-5, 4e-4, 4e-5
FRACAO_WARMUP, WEIGHT_DECAY, CLIP_GRAD = 0.10, 0.1, 1.0

In [ ]:
import csv
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
from importlib.metadata import version
from itertools import islice
from pathlib import Path
from zipfile import ZipFile


def esta_no_colab():
    try:
        from google import colab
    except ImportError:
        return False
    return colab is not None


def instalar_dependencias():
    if esta_no_colab():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "tokenizers==0.22.1", "tiktoken==0.11.0"],
            check=True,
        )


instalar_dependencias()

In [ ]:
FIM_DOC = "<|endoftext|>"
VERSAO_PIPELINE = 1


VOCABULARIO = CONFIG_VANILLA["vocab_size"]
CONTEXTO = CONFIG_VANILLA["context_length"]

import matplotlib.pyplot as plt
import numpy as np
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Markdown, display
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from torch.utils.data import DataLoader, Dataset

In [ ]:
def escolher_dispositivo():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Localizar o corpus

O treino extrai do ZIP os textos de treino e validação e o BPE de referência. No Colab, os dados extraídos ficam na máquina da sessão e os artefatos persistentes vão para o Drive. Para a inferência, basta montar o Drive e acessar os pesos e o BPE salvos.

In [ ]:
def sha256(arquivo):
    with arquivo.open("rb") as fonte:
        return hashlib.file_digest(fonte, "sha256").hexdigest()


def temporario(caminho):
    return caminho.with_name(caminho.name + ".tmp")


def gravar_json(caminho, dados):
    tmp = temporario(caminho)
    tmp.write_text(json.dumps(dados, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    tmp.replace(caminho)


def gravar_csv(caminho, linhas):
    tmp = temporario(caminho)
    with tmp.open("w", encoding="utf-8", newline="") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=list(linhas[0]))
        escritor.writeheader()
        escritor.writerows(linhas)
    tmp.replace(caminho)

In [ ]:
ARQUIVOS_CORPUS = ("train.txt", "val.txt", "tokenizer_16k/tokenizer.json")


def localizar_projeto():
    if esta_no_colab():
        from google.colab import drive

        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive") / PASTA_PROJETO_NO_DRIVE
    return Path("outputs")


def localizar_corpus():
    if not esta_no_colab():
        return Path("dataset/corpus_ufcg_introducoes_alunos")

    caminho_zip = Path("/content/drive/MyDrive") / ZIP_NO_DRIVE
    if not caminho_zip.is_file():
        raise FileNotFoundError(f"Coloque o ZIP oficial em {caminho_zip} ou ajuste ZIP_NO_DRIVE.")

    with ZipFile(caminho_zip) as pacote:
        membros = [pacote.getinfo(nome) for nome in ARQUIVOS_CORPUS]
        assinatura = "-".join(f"{membro.CRC:08x}" for membro in membros)
        pasta = Path("/content") / f"corpus_ufcg_{assinatura}"
        for membro in membros:
            destino = pasta / membro.filename
            if destino.is_file() and destino.stat().st_size == membro.file_size:
                continue
            destino.parent.mkdir(parents=True, exist_ok=True)
            with pacote.open(membro) as origem, temporario(destino).open("wb") as saida:
                shutil.copyfileobj(origem, saida)
            temporario(destino).replace(destino)
    return pasta

### Pastas dos experimentos

O BPE, a fertilidade e as segmentações ficam em `tokenizador/`. Cada modelo tem sua pasta de treino. O vanilla mantém `semana1_vanilla/` para poder retomar os checkpoints já gerados. A variante tem a pasta independente `modificado/`.

In [ ]:
def pastas_experimentos(pasta_projeto):
    return {
        "vanilla": pasta_projeto / "semana1_vanilla",
        "modificado": pasta_projeto / "modificado",
    }


def localizar_experimento(experimento, pasta_projeto):
    pastas = pastas_experimentos(pasta_projeto)
    if experimento not in pastas:
        raise ValueError(f"Experimento desconhecido: {experimento!r}.")
    return pastas[experimento]

## 3. O tokenizador

Treine um BPE em nível de byte com vocabulário de 16.000, usando apenas `train.txt`. O token especial `<|endoftext|>` separa os documentos. A classe `TokenizadorUFCG` mantém a interface de codificação e decodificação usada no livro.

In [ ]:
class TokenizadorUFCG:
    """Envolve o tokenizador da Hugging Face com a interface do tiktoken usada no livro."""
    def __init__(self, caminho):
        self.tok = Tokenizer.from_file(str(caminho))
        self.eot_id = self.tok.token_to_id("<|endoftext|>")
        self.n_vocab = self.tok.get_vocab_size()

    def encode(self, texto, allowed_special=None):
        # <|endoftext|> já é reconhecido como token especial pelo tokenizer.json
        return self.tok.encode(texto).ids

    def encode_batch(self, textos):
        return [e.ids for e in self.tok.encode_batch(textos)]

    def decode(self, ids):
        return self.tok.decode(list(ids), skip_special_tokens=False)

In [ ]:
def documentos(arquivo):
    def ler():
        partes = []
        with arquivo.open(encoding="utf-8") as fonte:
            for linha in fonte:
                if linha.rstrip("\n") == FIM_DOC:
                    texto = "".join(partes).removesuffix("\n")
                    if texto.strip():
                        yield texto
                    partes.clear()
                else:
                    partes.append(linha)
            texto = "".join(partes)
            if texto.strip():
                yield texto

    return ler()


def lotes_de_documentos(arquivo):
    fluxo = documentos(arquivo)
    while lote := list(islice(fluxo, 128)):
        yield lote

In [ ]:
def treinar_bpe(arquivo_treino):
    bpe = Tokenizer(models.BPE())
    bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    bpe.decoder = decoders.ByteLevel()
    treinador = trainers.BpeTrainer(
        vocab_size=VOCABULARIO,
        min_frequency=2,
        special_tokens=[FIM_DOC],
        initial_alphabet=sorted(pre_tokenizers.ByteLevel.alphabet()),
        show_progress=False,
    )
    bpe.train_from_iterator(lotes_de_documentos(arquivo_treino), treinador)
    if bpe.get_vocab_size() != VOCABULARIO:
        raise ValueError(f"O BPE aprendeu {bpe.get_vocab_size()} IDs; esperávamos {VOCABULARIO}.")
    return bpe

In [ ]:
def carregar_ou_treinar_bpe(arquivo_treino, hash_treino, pasta_projeto):
    pasta_tokenizador = pasta_projeto / "tokenizador"
    pasta_tokenizador.mkdir(parents=True, exist_ok=True)
    destino = pasta_tokenizador / "tokenizer.json"
    metadados = pasta_tokenizador / "tokenizer_config.json"
    pastas = pastas_experimentos(pasta_projeto)
    config = {
        "pipeline": VERSAO_PIPELINE,
        "train_sha256": hash_treino,
        "limite_documentos": None,
        "vocabulario": VOCABULARIO,
        "tokenizers": version("tokenizers"),
    }

    for pasta in (pasta_tokenizador, pastas["vanilla"]):
        caminho = pasta / destino.name
        metadados_origem = pasta / metadados.name
        if not (caminho.is_file() and metadados_origem.is_file()):
            continue
        anterior = json.loads(metadados_origem.read_text(encoding="utf-8"))
        if anterior == {**config, "tokenizer_sha256": sha256(caminho)}:
            if caminho != destino:
                shutil.copyfile(caminho, temporario(destino))
                temporario(destino).replace(destino)
                gravar_json(metadados, anterior)
            return TokenizadorUFCG(destino)

    if any((pasta / "checkpoint.pt").exists() for pasta in pastas.values()):
        raise ValueError("O BPE não corresponde à configuração salva. Restaure-o ou use outra pasta de projeto.")

    bpe = treinar_bpe(arquivo_treino)
    bpe.save(str(temporario(destino)))
    temporario(destino).replace(destino)
    gravar_json(metadados, {**config, "tokenizer_sha256": sha256(destino)})
    return TokenizadorUFCG(destino)

In [ ]:
def preparar_tokenizadores(arquivo_treino, arquivo_referencia, assinatura, pasta_projeto, pasta_cache):
    tokenizer = carregar_ou_treinar_bpe(arquivo_treino, assinatura["train.txt"], pasta_projeto)
    referencia = Tokenizer.from_file(str(arquivo_referencia))
    os.environ.setdefault("TIKTOKEN_CACHE_DIR", str(pasta_cache / "tiktoken"))
    gpt2_tiktoken = tiktoken.get_encoding("gpt2")
    assert tokenizer.encode(FIM_DOC) == [tokenizer.eot_id]
    print(f"BPE próprio: {tokenizer.n_vocab:,} IDs | fim de documento: {tokenizer.eot_id}")
    return tokenizer, referencia, gpt2_tiktoken

In [ ]:
EXEMPLO = "Este trabalho tem como objetivo analisá-lo à luz da literatura."


def demonstrar_tokenizador(tokenizer):
    ids = tokenizer.encode(EXEMPLO)
    print("IDs:", ids)
    print("Segmentação:", tokenizer.tok.encode(EXEMPLO).tokens)
    assert tokenizer.decode(ids) == EXEMPLO
    print("Decodificado:", tokenizer.decode(ids))

**Fertilidade** é o número de tokens por palavra. Comparamos o BPE próprio, o BPE de referência e o `gpt2` do `tiktoken` no mesmo texto de treino, descontando os marcadores de fim de documento. A fertilidade de referência informada no enunciado é de aproximadamente 1,37.

In [ ]:
def medir_fertilidade(arquivo_treino, tokenizer, referencia, gpt2_tiktoken):
    totais = dict.fromkeys(("BPE próprio", "BPE de referência", "GPT-2 tiktoken"), 0)
    palavras, n_documentos = 0, 0
    for lote in lotes_de_documentos(arquivo_treino):
        n_documentos += len(lote)
        palavras += sum(len(texto.split()) for texto in lote)
        totais["BPE próprio"] += sum(len(ids) for ids in tokenizer.encode_batch(lote))
        totais["BPE de referência"] += sum(len(x.ids) for x in referencia.encode_batch(lote))
        totais["GPT-2 tiktoken"] += sum(len(x) for x in gpt2_tiktoken.encode_ordinary_batch(lote))
    return [
        {"tokenizador": nome, "documentos": n_documentos, "palavras": palavras,
         "tokens": total, "tokens_por_palavra": total / palavras}
        for nome, total in totais.items()
    ]

In [ ]:
def carregar_ou_medir_fertilidade(arquivo_treino, tokenizadores, assinatura, pasta_tokenizador):
    caminho = pasta_tokenizador / "fertilidade.csv"
    metadados = pasta_tokenizador / "fertilidade_config.json"
    config = {
        "tokenizer_sha256": sha256(pasta_tokenizador / "tokenizer.json"),
        "referencia_sha256": assinatura["tokenizer_16k/tokenizer.json"],
        "train_sha256": assinatura["train.txt"],
        "limite_documentos": None,
        "tiktoken": version("tiktoken"),
        "pipeline": VERSAO_PIPELINE,
    }
    if caminho.is_file() and metadados.is_file():
        if json.loads(metadados.read_text(encoding="utf-8")) == config:
            with caminho.open(encoding="utf-8", newline="") as arquivo:
                return list(csv.DictReader(arquivo))

    resultados = medir_fertilidade(arquivo_treino, *tokenizadores)
    gravar_csv(caminho, resultados)
    gravar_json(metadados, config)
    return resultados

In [ ]:
def exibir_fertilidade(resultados):
    linhas = ["| Tokenizador | Documentos | Tokens por palavra |", "|---|---:|---:|"]
    for linha in resultados:
        linhas.append(
            f"| {linha['tokenizador']} | {linha['documentos']} | {float(linha['tokens_por_palavra']):.4f} |"
        )
    display(Markdown("\n".join(linhas)))
    diferenca = float(resultados[1]["tokens_por_palavra"]) - 1.37
    print(f"Diferença do BPE de referência para os 1,37 esperados: {diferenca:+.4f}.")

In [ ]:
EXEMPLOS_SEGMENTACAO = ("ação", "informações", "dá-lo", "analisá-lo", "desenvolveram", "compreenderíamos")


def comparar_segmentacoes(tokenizadores, pasta_tokenizador):
    tokenizer, referencia, gpt2_tiktoken = tokenizadores
    segmentacoes = []
    for palavra in EXEMPLOS_SEGMENTACAO:
        ids_gpt2 = gpt2_tiktoken.encode_ordinary(palavra)
        segmentacoes.append({
            "palavra": palavra,
            "bpe_proprio_tokens": tokenizer.tok.encode(palavra).tokens,
            "bpe_referencia_tokens": referencia.encode(palavra).tokens,
            "gpt2_pedacos_bytes": [repr(gpt2_tiktoken.decode_single_token_bytes(i)) for i in ids_gpt2],
        })
        assert tokenizer.decode(tokenizer.encode(palavra)) == palavra

    gravar_csv(pasta_tokenizador / "segmentacoes.csv", segmentacoes)
    linhas = ["| Texto | BPE próprio | Referência | GPT-2 tiktoken |", "|---|---|---|---|"]
    for linha in segmentacoes:
        linhas.append(
            f"| {linha['palavra']} | `{linha['bpe_proprio_tokens']}` | "
            f"`{linha['bpe_referencia_tokens']}` | `{linha['gpt2_pedacos_bytes']}` |"
        )
    display(Markdown("\n".join(linhas)))

## 4. Pré-tokenização (uma vez só)

Cada documento é codificado separadamente e recebe `<|endoftext|>` no fim. Salvamos os IDs em arquivos `.npy` com `uint16`. O cache usa as assinaturas do corpus e do BPE, de forma que outro tokenizador gere seu próprio arquivo.

In [ ]:
def pre_tokenizar(arquivo, tokenizer, pasta_cache, hash_bpe, hash_texto):
    chave = f"v{VERSAO_PIPELINE}-{hash_bpe[:16]}-{hash_texto[:16]}-None"
    destino = pasta_cache / f"{arquivo.stem}-{chave}.npy"
    if not destino.exists():
        blocos = []
        for lote in lotes_de_documentos(arquivo):
            ids_lote = []
            for ids in tokenizer.encode_batch(lote):
                ids_lote.extend(ids)
                ids_lote.append(tokenizer.eot_id)
            blocos.append(np.asarray(ids_lote, dtype=np.uint16))
        ids = np.concatenate(blocos)
        with temporario(destino).open("wb") as saida:
            np.save(saida, ids, allow_pickle=False)
        temporario(destino).replace(destino)
    return np.load(destino, mmap_mode="r", allow_pickle=False)

In [ ]:
def pre_tokenizar_particoes(pasta_corpus, tokenizer, pasta_cache, hash_bpe, assinatura):
    t0 = time.perf_counter()
    ids_treino, ids_val = (
        pre_tokenizar(pasta_corpus / nome, tokenizer, pasta_cache, hash_bpe, assinatura[nome])
        for nome in ("train.txt", "val.txt")
    )
    print(f"Treino: {len(ids_treino):,} tokens | Validação: {len(ids_val):,} tokens | "
          f"{time.perf_counter() - t0:.1f} s")
    return ids_treino, ids_val

## 5. Dataset: a janela deslizante do livro, sem estourar a memória

Cada alvo corresponde à entrada deslocada uma posição. Mantemos o dataset do livro como conferência em um exemplo curto. No treino e na validação, recortamos as janelas sob demanda a partir do array de IDs. A última janela parcial da validação usa preenchimento ignorado na loss.

In [ ]:
class GPTDatasetV1(Dataset):
    """Cópia do GPTDatasetV1 do livro (cap. 2), usada só para conferência num trecho pequeno."""
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids, self.target_ids = [], []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
class GPTDatasetTokens(Dataset):
    def __init__(self, ids, contexto, fim_documento, manter_cauda=False, *, stride=None):
        self.ids = ids
        self.contexto = contexto
        self.fim_documento = fim_documento
        self.stride = contexto if stride is None else stride
        if contexto <= 0 or self.stride <= 0 or (manter_cauda and self.stride != contexto):
            raise ValueError("Use contexto e stride positivos e janelas contíguas com cauda.")
        n_alvos = max(0, len(ids) - 1)
        self.n_janelas = max(0, 1 + (n_alvos - contexto) // self.stride)
        if manter_cauda and n_alvos % contexto:
            self.n_janelas += 1

    def __len__(self):
        return self.n_janelas

    def __getitem__(self, indice):
        inicio = indice * self.stride
        janela = self.ids[inicio:inicio + self.contexto + 1]
        n = len(janela) - 1
        x = np.full(self.contexto, self.fim_documento, dtype=np.int64)
        y = np.full(self.contexto, -100, dtype=np.int64)
        x[:n], y[:n] = janela[:-1], janela[1:]
        return torch.from_numpy(x), torch.from_numpy(y)

In [ ]:
def conferir_janelas(tokenizer):
    # Conferência em um exemplo didático, sem exibir trechos do corpus privado.
    trecho = (EXEMPLO + " ") * 4
    ds_livro = GPTDatasetV1(trecho, tokenizer, max_length=8, stride=4)
    ds_nosso = GPTDatasetTokens(np.asarray(tokenizer.encode(trecho), dtype=np.uint16),
                                contexto=8, fim_documento=tokenizer.eot_id, stride=4)
    assert len(ds_livro) == len(ds_nosso) > 0
    for indice in range(len(ds_livro)):
        x_livro, y_livro = ds_livro[indice]
        x_nosso, y_nosso = ds_nosso[indice]
        assert torch.equal(x_livro, x_nosso) and torch.equal(y_livro, y_nosso)
    print(f"OK: {len(ds_livro)} janelas idênticas às do livro.")

## 6. DataLoaders

Os lotes de treino seguem uma ordem embaralhada com semente fixa. Ao retomar, usamos a mesma ordem a partir do próximo lote. No treino oficial, o stride é igual ao contexto e o orçamento é uma passagem pelas janelas de lotes completos. A validação conserva a última janela parcial.

In [ ]:
def criar_dataloader(dataset, lote, seed, dispositivo, *, indices=None):
    return DataLoader(
        dataset, batch_size=lote, sampler=indices, num_workers=0,
        pin_memory=dispositivo.type == "cuda",
        generator=torch.Generator().manual_seed(seed),
    )


def preparar_dataloaders(ids_treino, ids_val, fim_documento, dispositivo):
    dataset_treino = GPTDatasetTokens(ids_treino, CONTEXTO, fim_documento)
    dataset_val = GPTDatasetTokens(ids_val, CONTEXTO, fim_documento, manter_cauda=True)
    passos_totais = len(dataset_treino) // LOTE
    if passos_totais < 2 or len(dataset_val) == 0:
        raise ValueError("Corpus insuficiente para o orçamento e o lote escolhidos.")

    loader_val = criar_dataloader(dataset_val, LOTE, SEMENTE, dispositivo)
    gerador_ordem = torch.Generator().manual_seed(SEMENTE)
    ordem = torch.randperm(len(dataset_treino), generator=gerador_ordem)
    ordem_treino = ordem[:passos_totais * LOTE].tolist()
    tokens_por_lote = LOTE * CONTEXTO
    print(f"{passos_totais:,} passos × {tokens_por_lote:,} tokens = "
          f"{passos_totais * tokens_por_lote:,} tokens de treino")
    return dataset_treino, loader_val, ordem_treino, passos_totais

In [ ]:
def conferir_lote(dataset_treino, ordem_treino, dispositivo):
    loader = criar_dataloader(dataset_treino, LOTE, SEMENTE, dispositivo, indices=ordem_treino[:LOTE])
    x, y = next(iter(loader))
    assert x.shape == y.shape == (LOTE, CONTEXTO)
    assert torch.equal(x[:, 1:], y[:, :-1])
    print(f"Entrada: {tuple(x.shape)} | Alvo: {tuple(y.shape)} | dtype: {x.dtype}")
    print("Alvo deslocado de uma posição: OK.")
    return x, y

## 7. Conferência com o `GPTModel` do livro

O vanilla preserva a arquitetura dos capítulos 3 e 4 de Sebastian Raschka, incluindo atenção causal, posições aprendidas, LayerNorm e GELU. A configuração do topo reduz o modelo para as dimensões da atividade. As primeiras conferências verificam formatos, causalidade, parâmetros e loss inicial.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "A dimensão deve ser divisível pelo número de cabeças."

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, _ = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            math.sqrt(2.0 / math.pi) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

### Modelos registrados

O vanilla é a referência da Semana 1. Quando a técnica da variante for escolhida, registre a classe e sua configuração com `MODELOS["modificado"] = (GPTModificado, config_modificado)`. Os dois modelos usam o corpus, o BPE e o orçamento definidos acima.

In [ ]:
MODELOS = {"vanilla": (GPTModel, CONFIG_VANILLA)}


def definir_modelo(experimento):
    if experimento not in ("vanilla", "modificado"):
        raise ValueError(f"Experimento desconhecido: {experimento!r}.")
    if experimento not in MODELOS:
        raise ValueError("Registre a arquitetura escolhida para a variante na seção Modelos registrados.")
    fabrica, config_modelo = MODELOS[experimento]
    if config_modelo["vocab_size"] != VOCABULARIO or config_modelo["context_length"] != CONTEXTO:
        raise ValueError("Os experimentos devem usar o vocabulário e o contexto do vanilla.")
    return fabrica, config_modelo


@torch.inference_mode()
def conferir_modelo(modelo, dispositivo):
    modelo.eval()
    prefixos = torch.tensor([[1, 2, 3, 4], [1, 2, 5, 6]], device=dispositivo)
    saida = modelo(prefixos)
    assert saida.shape == (2, 4, VOCABULARIO)
    torch.testing.assert_close(saida[0, :2], saida[1, :2])
    modelo.train()

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device, non_blocking=True)
    target_batch = target_batch.to(device, non_blocking=True)
    logits = model(input_batch)
    return F.cross_entropy(logits.flatten(0, 1), target_batch.flatten())


@torch.inference_mode()
def conferir_vanilla_inicial(modelo, x, y, dispositivo):
    modelo.eval()
    total = sum(p.numel() for p in modelo.parameters())
    loss = calc_loss_batch(x[:1], y[:1], modelo, dispositivo).item()
    print(f"Parâmetros: {total:,} | Loss inicial: {loss:.2f} "
          f"(referência uniforme: {math.log(VOCABULARIO):.2f}).")

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size, eos_id=None):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)
        if eos_id is not None and (idx_next == eos_id).all():
            break

    return idx

In [ ]:
@torch.inference_mode()
def demonstrar_modelo_inicial(modelo, tokenizer, dispositivo):
    # Geração com pesos aleatórios, apenas para conferir o caminho de inferência.
    entrada = torch.tensor([tokenizer.encode("Este trabalho tem como objetivo")], device=dispositivo)
    gerados = generate_text_simple(modelo, entrada, max_new_tokens=6, context_size=CONTEXTO)
    print("Exemplo antes do treino:", tokenizer.decode(gerados[0].tolist()))

### Loop de treino do Apêndice D

O vanilla oficial treina por uma época. Usamos AdamW com decay nas matrizes, warmup linear, cosine decay, gradient clipping e FP16 com `GradScaler`. Os lotes contam no orçamento, e o scheduler avança quando há uma atualização efetiva dos pesos.

In [ ]:
def configurar_treino(experimento, config_modelo, assinatura, hash_bpe, ids_treino, passos_totais):
    passos_warmup = min(passos_totais - 1, max(1, round(passos_totais * FRACAO_WARMUP)))
    return {
        "experimento": experimento,
        "pipeline": VERSAO_PIPELINE,
        "modo": "oficial",
        "modelo": config_modelo,
        "corpus_sha256": assinatura,
        "tokenizer_sha256": hash_bpe,
        "seed": SEMENTE,
        "lote": LOTE,
        "passos": passos_totais,
        "tokens_planejados": passos_totais * LOTE * CONTEXTO,
        "tokens_disponiveis_treino": len(ids_treino) - 1,
        "passos_warmup": passos_warmup,
        "lr_inicial": LR_INICIAL,
        "lr_pico": LR_PICO,
        "lr_minimo": LR_MINIMO,
        "weight_decay": WEIGHT_DECAY,
        "clip_grad": CLIP_GRAD,
        "precisao": "fp16",
        "documentos_treino_limite": None,
        "documentos_val_limite": None,
    }


def fator_lr(atualizacao, config):
    warmup = config["passos_warmup"]
    inicial, pico, minimo = config["lr_inicial"], config["lr_pico"], config["lr_minimo"]
    if atualizacao < warmup:
        lr = inicial + (pico - inicial) * atualizacao / warmup
    else:
        progresso = min(1.0, (atualizacao - warmup) / max(1, config["passos"] - 1 - warmup))
        lr = minimo + (pico - minimo) * (1 + math.cos(math.pi * progresso)) / 2
    return lr / pico


def configurar_otimizacao(modelo, config):
    decay = [p for p in modelo.parameters() if p.ndim >= 2]
    sem_decay = [p for p in modelo.parameters() if p.ndim < 2]
    otimizador = torch.optim.AdamW(
        [{"params": decay, "weight_decay": config["weight_decay"]},
         {"params": sem_decay, "weight_decay": 0.0}],
        lr=config["lr_pico"],
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(otimizador, lambda passo: fator_lr(passo, config))
    scaler = torch.amp.GradScaler("cuda", init_scale=1024, enabled=config["precisao"] == "fp16")
    return otimizador, scheduler, scaler

In [ ]:
def treinar_lote(modelo, lote, otimizador, scheduler, scaler, estado, config):
    dispositivo_modelo = next(modelo.parameters()).device
    lr = otimizador.param_groups[0]["lr"]
    otimizador.zero_grad(set_to_none=True)
    with autocast(dispositivo_modelo, scaler.is_enabled()):
        loss = calc_loss_batch(lote[0], lote[1], modelo, dispositivo_modelo)
    scaler.scale(loss).backward()
    if estado["atualizacoes"] >= config["passos_warmup"]:
        scaler.unscale_(otimizador)
        nn.utils.clip_grad_norm_(
            modelo.parameters(), config["clip_grad"], error_if_nonfinite=not scaler.is_enabled()
        )

    escala_anterior = scaler.get_scale()
    scaler.step(otimizador)
    scaler.update()
    if scaler.get_scale() >= escala_anterior:
        estado["atualizacoes"] += 1
        scheduler.step()
    return loss.detach().item(), lr

In [ ]:
def treinar_modelo(
    modelo, config, pasta, criar_loader, avaliar_validacao, *, intervalo,
):
    dispositivo_modelo = next(modelo.parameters()).device
    fp16 = config["precisao"] == "fp16"
    otimizador, scheduler, scaler = configurar_otimizacao(modelo, config)
    caminho_checkpoint = pasta / "checkpoint.pt"
    caminho_metricas = pasta / "metricas.csv"
    estado = recuperar_estado(caminho_checkpoint, config, modelo, otimizador, scheduler, scaler)

    gravar_json(pasta / "configuracao.json", config)
    inicio_parede = time.perf_counter()
    parede_anterior = estado["parede_s"]

    def tempo_de_parede():
        return parede_anterior + time.perf_counter() - inicio_parede

    if not estado["historico"]:
        metricas = avaliar_validacao(modelo)
        registrar_validacao(estado, config, metricas, otimizador.param_groups[0]["lr"], tempo_de_parede())
    gravar_csv(caminho_metricas, estado["historico"])

    fim = config["passos"]
    if estado["passo"] >= fim:
        return estado

    n_tokens = config["lote"] * config["modelo"]["context_length"]
    marcos = {math.ceil(config["passos"] * i / 10) for i in range(1, 11)}
    loader = criar_loader(estado["passo"], fim)
    modelo.train()
    if fp16:
        torch.cuda.reset_peak_memory_stats(dispositivo_modelo)

    inicio_lote = time.perf_counter()
    for passo, lote in enumerate(loader, start=estado["passo"] + 1):
        loss, lr = treinar_lote(modelo, lote, otimizador, scheduler, scaler, estado, config)
        if not math.isfinite(loss):
            raise FloatingPointError(f"Loss não finita no passo {passo}.")
        estado["treino_s"] += time.perf_counter() - inicio_lote
        estado["nll_intervalo"] += loss * n_tokens
        estado["tokens_intervalo"] += n_tokens
        estado["passo"] = passo
        if fp16:
            pico = torch.cuda.max_memory_allocated(dispositivo_modelo)
            estado["pico_gpu_bytes"] = max(estado["pico_gpu_bytes"], pico)

        if passo in marcos:
            metricas = avaliar_validacao(modelo)
            registrar_validacao(estado, config, metricas, lr, tempo_de_parede())
            gravar_csv(caminho_metricas, estado["historico"])
            if fp16:
                torch.cuda.reset_peak_memory_stats(dispositivo_modelo)
        if passo % intervalo == 0 or passo == fim:
            estado["parede_s"] = tempo_de_parede()
            salvar_checkpoint(caminho_checkpoint, config, modelo, otimizador, scheduler, scaler, estado)
        inicio_lote = time.perf_counter()
    return estado

## 8. Checkpoint e retomada

O checkpoint salva pesos, otimizador, scheduler, scaler, estado aleatório, histórico e posição do lote. O treino pode continuar com a mesma configuração depois de uma desconexão. `semana1_vanilla/` guarda o baseline e `modificado/` fica reservada para a variante. O BPE é compartilhado.

In [ ]:
def ler_checkpoint(caminho):
    checkpoint = torch.load(caminho, map_location="cpu", weights_only=True)
    checkpoint["config"].setdefault("experimento", "vanilla")
    return checkpoint


def salvar_checkpoint(caminho, config, modelo, otimizador, scheduler, scaler, estado):
    dispositivo_modelo = next(modelo.parameters()).device
    checkpoint = {
        "config": config,
        "modelo": modelo.state_dict(),
        "otimizador": otimizador.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "estado": estado,
        "rng_cpu": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state(dispositivo_modelo) if scaler.is_enabled() else None,
    }
    tmp = temporario(caminho)
    torch.save(checkpoint, tmp)
    tmp.replace(caminho)


def recuperar_estado(caminho, config, modelo, otimizador, scheduler, scaler):
    if not caminho.exists():
        return dict(passo=0, atualizacoes=0, nll_intervalo=0.0, tokens_intervalo=0,
                    treino_s=0.0, parede_s=0.0, pico_gpu_bytes=0, historico=[])

    checkpoint = ler_checkpoint(caminho)
    if checkpoint["config"] != config:
        raise ValueError(f"Configuração diferente da gravada em {caminho}. Use outra pasta de projeto.")
    modelo.load_state_dict(checkpoint["modelo"])
    otimizador.load_state_dict(checkpoint["otimizador"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    torch.set_rng_state(checkpoint["rng_cpu"])
    if scaler.is_enabled():
        torch.cuda.set_rng_state(checkpoint["rng_cuda"], next(modelo.parameters()).device)
    estado = checkpoint["estado"]
    print(f"Retomando {config['experimento']} após o passo {estado['passo']:,}/{config['passos']:,}.")
    return estado

In [ ]:
def carregar_modelo_e_bpe(experimento):
    fabrica, config_modelo = definir_modelo(experimento)
    pasta_projeto = localizar_projeto()
    pasta = localizar_experimento(experimento, pasta_projeto)
    caminho = pasta / "checkpoint.pt"
    if not caminho.is_file():
        raise FileNotFoundError(f"Coloque o checkpoint treinado em {caminho}.")

    checkpoint = ler_checkpoint(caminho)
    config_salva = checkpoint["config"]
    if config_salva["experimento"] != experimento or config_salva["modelo"] != config_modelo:
        raise ValueError(f"A arquitetura registrada não corresponde ao checkpoint de {experimento}.")

    candidatos = (pasta_projeto / "tokenizador" / "tokenizer.json", pasta / "tokenizer.json")
    caminho_bpe = next(
        (p for p in candidatos if p.is_file() and sha256(p) == config_salva["tokenizer_sha256"]),
        None,
    )
    if caminho_bpe is None:
        raise FileNotFoundError(f"Restaure o tokenizer.json original em {candidatos[0]}.")

    modelo = fabrica(config_modelo).to(escolher_dispositivo())
    modelo.load_state_dict(checkpoint["modelo"])
    return modelo.eval(), TokenizadorUFCG(caminho_bpe)


def carregar_modelo(experimento):
    modelo, _ = carregar_modelo_e_bpe(experimento)
    return modelo

### Gerar a partir de um checkpoint

Em uma sessão nova, execute as células desde o início até a função `gerar_com_checkpoint` abaixo. Em seguida, use uma célula com `print(gerar_com_checkpoint("vanilla", "Este trabalho tem como objetivo"))`. A função carrega o checkpoint e o BPE salvos. Após registrar e treinar a variante, use `"modificado"` na mesma chamada.

In [ ]:
@torch.inference_mode()
def gerar_com_checkpoint(experimento, prompt, max_novos_tokens=30):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("Informe um prompt de texto.")
    if not isinstance(max_novos_tokens, int) or max_novos_tokens < 1:
        raise ValueError("Informe um número inteiro positivo de novos tokens.")

    modelo, tokenizer = carregar_modelo_e_bpe(experimento)
    dispositivo = next(modelo.parameters()).device
    ids_prompt = tokenizer.encode(prompt)
    entrada = torch.tensor([ids_prompt], dtype=torch.long, device=dispositivo)
    saida = generate_text_simple(
        modelo, entrada, max_new_tokens=max_novos_tokens,
        context_size=CONTEXTO, eos_id=tokenizer.eot_id,
    )
    ids = saida[0].tolist()
    if ids[-1] == tokenizer.eot_id:
        ids.pop()
    return tokenizer.decode(ids)

## 9. Bits por byte na validação

Calculamos BPB dividindo a soma da loss em nats por `ln(2) × número de bytes UTF-8 dos alvos de texto`. Marcadores de fim de documento e preenchimento ficam fora dessa conta. Os símbolos ByteLevel representam os bytes originais mesmo quando um token contém só parte de um caractere. A loss de validação e a perplexidade usam todos os alvos válidos, incluindo o marcador de fim de documento.

In [ ]:
def contar_bytes_do_bpe(tokenizer, dispositivo):
    larguras = [0] * tokenizer.n_vocab
    for simbolos, indice in tokenizer.tok.get_vocab().items():
        if indice != tokenizer.eot_id:
            larguras[indice] = len(simbolos)
    for palavra in EXEMPLOS_SEGMENTACAO:
        assert sum(larguras[i] for i in tokenizer.encode(palavra)) == len(palavra.encode("utf-8"))
    return torch.tensor(larguras, device=dispositivo)


def autocast(dispositivo, fp16):
    return torch.autocast(dispositivo.type, dtype=torch.float16, enabled=fp16)


@torch.inference_mode()
def avaliar(modelo, loader, bytes_por_id, fim_documento, *, fp16):
    dispositivo = bytes_por_id.device
    estava_treinando = modelo.training
    modelo.eval()
    nll_tokens = nll_texto = 0.0
    n_tokens = n_bytes = 0
    try:
        for x, y in loader:
            x = x.to(dispositivo, non_blocking=True)
            y = y.to(dispositivo, non_blocking=True).flatten()
            with autocast(dispositivo, fp16):
                logits = modelo(x).flatten(0, 1)
                perdas = F.cross_entropy(logits, y, ignore_index=-100, reduction="none")
            validos = y != -100
            texto = validos & (y != fim_documento)
            nll_tokens += perdas.sum(dtype=torch.float64).item()
            nll_texto += perdas[texto].sum(dtype=torch.float64).item()
            n_tokens += validos.sum().item()
            n_bytes += bytes_por_id[y[texto]].sum().item()
        if n_tokens == 0 or n_bytes == 0 or not math.isfinite(nll_tokens):
            raise ValueError("A avaliação precisa de tokens de texto e loss finita.")
        loss = nll_tokens / n_tokens
        return {"loss_val": loss, "perplexidade_val": math.exp(loss),
                "bpb_val": nll_texto / (math.log(2) * n_bytes),
                "tokens_val": n_tokens, "bytes_val": n_bytes}
    finally:
        modelo.train(estava_treinando)

In [ ]:
def registrar_validacao(estado, config, metricas, lr, parede_s):
    passo = estado["passo"]
    tokens = passo * config["lote"] * config["modelo"]["context_length"]
    n = estado["tokens_intervalo"]
    linha = {
        "passo": passo, "tokens_treino": tokens, "learning_rate": lr,
        "loss_treino": estado["nll_intervalo"] / n if n else None,
        **metricas,
        "tokens_por_segundo": tokens / estado["treino_s"] if tokens else None,
        "treino_s": estado["treino_s"],
        "parede_s": parede_s,
        "pico_gpu_mib": estado["pico_gpu_bytes"] / 2**20 if config["precisao"] == "fp16" else None,
        "atualizacoes_ignoradas": passo - estado["atualizacoes"],
    }
    estado["historico"].append(linha)
    estado.update(nll_intervalo=0.0, tokens_intervalo=0)
    print(f"Passo {passo:,}/{config['passos']:,} | loss val {linha['loss_val']:.4f} | BPB {linha['bpb_val']:.4f}")

## 10. Conclusão da Semana 1

A execução percorre as etapas na ordem do roteiro: prepara os dados, compara os tokenizadores, confere o vanilla e inicia ou retoma o treino. Ao concluir, salva a curva, o BPB em `val.txt`, as métricas e os checkpoints. Os CSVs de fertilidade e segmentação ficam na pasta do BPE compartilhado.

In [ ]:
def plotar_curvas(historico, experimento, pasta):
    pontos_treino = [linha for linha in historico if linha["loss_treino"] is not None]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([linha["tokens_treino"] for linha in pontos_treino],
            [linha["loss_treino"] for linha in pontos_treino], "o-", label="Treino")
    ax.plot([linha["tokens_treino"] for linha in historico],
            [linha["loss_val"] for linha in historico], "o-", label="Validação")
    ax.set(xlabel="Tokens de treino processados", ylabel="Loss (nats/token)",
           title=f"GPT-2 {experimento}")
    ax.ticklabel_format(axis="x", style="plain")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(pasta / f"curvas_{experimento}.png", dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
def exibir_resumo(resumo, pasta):
    campos = [
        ("Tokens de treino", "tokens_treino", ",d"),
        ("Loss de treino no último intervalo", "loss_treino", ".4f"),
        ("Loss de validação", "loss_val", ".4f"),
        ("Perplexidade de validação", "perplexidade_val", ".2f"),
        ("Bits por byte de validação", "bpb_val", ".4f"),
        ("Parâmetros totais", "parametros_totais", ",d"),
        ("Parâmetros ativos por token", "parametros_ativos_por_token", ",d"),
        ("Throughput de treino (tokens/s)", "tokens_por_segundo", ",.1f"),
        ("Pico de memória GPU (MiB)", "pico_gpu_mib", ",.1f"),
        ("Tempo de parede (s)", "parede_s", ",.1f"),
    ]
    linhas = [
        f"**Execução oficial · {resumo['experimento']} · {resumo['dispositivo']} · {resumo['precisao']}**",
        "", "| Métrica | Valor |", "|---|---:|",
    ]
    for titulo, chave, formato in campos:
        valor = resumo[chave]
        linhas.append(f"| {titulo} | {format(valor, formato) if valor is not None else '—'} |")
    linhas.extend(["", f"Arquivos em `{pasta}`."])
    display(Markdown("\n".join(linhas)))


def publicar_resultados(modelo, estado, config, pasta):
    experimento = config["experimento"]
    parametros = sum(p.numel() for p in modelo.parameters())
    resumo = {
        "experimento": experimento,
        "modo": config["modo"],
        "dispositivo": torch.cuda.get_device_name(next(modelo.parameters()).device),
        "precisao": config["precisao"],
        "torch": str(torch.__version__),
        "parametros_totais": parametros,
        "parametros_ativos_por_token": parametros,
        **estado["historico"][-1],
        "parede_s": estado["parede_s"],
        "checkpoint_sha256": sha256(pasta / "checkpoint.pt"),
    }
    gravar_json(pasta / "resumo.json", resumo)
    plotar_curvas(estado["historico"], experimento, pasta)
    exibir_resumo(resumo, pasta)
    return resumo

In [ ]:
def preparar_dados(pasta_projeto):
    corpus = localizar_corpus()
    pasta_cache = Path("/content/cache_tokens") if esta_no_colab() else pasta_projeto / "cache_tokens"
    pasta_cache.mkdir(parents=True, exist_ok=True)
    print(f"Corpus: {corpus} | Projeto: {pasta_projeto}")

    assinatura = {nome: sha256(corpus / nome) for nome in ARQUIVOS_CORPUS}
    arquivo_treino, _, arquivo_referencia = (corpus / nome for nome in ARQUIVOS_CORPUS)
    tokenizadores = preparar_tokenizadores(arquivo_treino, arquivo_referencia, assinatura, pasta_projeto, pasta_cache)
    tokenizer, _, _ = tokenizadores
    demonstrar_tokenizador(tokenizer)
    pasta_tokenizador = pasta_projeto / "tokenizador"
    exibir_fertilidade(carregar_ou_medir_fertilidade(arquivo_treino, tokenizadores, assinatura, pasta_tokenizador))
    comparar_segmentacoes(tokenizadores, pasta_tokenizador)

    hash_bpe = sha256(pasta_tokenizador / "tokenizer.json")
    ids_treino, ids_val = pre_tokenizar_particoes(corpus, tokenizer, pasta_cache, hash_bpe, assinatura)
    conferir_janelas(tokenizer)
    return tokenizer, ids_treino, ids_val, assinatura, hash_bpe

In [ ]:
def executar_experimento(experimento):
    '''Treina ou retoma o modelo e apresenta os resultados quando concluído.'''
    fabrica, config_modelo = definir_modelo(experimento)
    dispositivo = escolher_dispositivo()
    if dispositivo.type != "cuda":
        raise RuntimeError("Ative uma GPU CUDA no Colab para executar o treino oficial.")

    pasta_projeto = localizar_projeto()
    tokenizer, ids_treino, ids_val, assinatura, hash_bpe = preparar_dados(pasta_projeto)
    dataset_treino, loader_val, ordem_treino, passos_totais = preparar_dataloaders(
        ids_treino, ids_val, tokenizer.eot_id, dispositivo,
    )
    x, y = conferir_lote(dataset_treino, ordem_treino, dispositivo)
    config = configurar_treino(experimento, config_modelo, assinatura, hash_bpe, ids_treino, passos_totais)
    pasta = localizar_experimento(experimento, pasta_projeto)
    pasta.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(config["seed"])
    modelo = fabrica(config_modelo).to(dispositivo)
    conferir_modelo(modelo, dispositivo)
    conferir_vanilla_inicial(modelo, x, y, dispositivo)
    demonstrar_modelo_inicial(modelo, tokenizer, dispositivo)
    modelo.train()
    bytes_por_id = contar_bytes_do_bpe(tokenizer, dispositivo)

    def criar_loader(inicio, fim):
        lote = config["lote"]
        indices = ordem_treino[inicio * lote:fim * lote]
        return criar_dataloader(dataset_treino, lote, config["seed"], dispositivo, indices=indices)

    def avaliar_validacao(modelo):
        return avaliar(modelo, loader_val, bytes_por_id, tokenizer.eot_id, fp16=True)

    estado = treinar_modelo(
        modelo, config, pasta, criar_loader, avaliar_validacao, intervalo=INTERVALO_CHECKPOINT,
    )
    return publicar_resultados(modelo, estado, config, pasta)

In [ ]:
resumo_execucao = executar_experimento("vanilla")

### Semanas seguintes

A Semana 2 acrescenta as estratégias de decodificação e a arquitetura modificada. Na Semana 3, reutilizamos o BPE, a semente e o orçamento do vanilla para treinar e comparar a variante. Guarde os dois checkpoints para a entrega final.